<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/Part2_7_DataAugmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2, Notebook 7, Data Variations

Two data side investigations, single channel versus three channels, and a moderate augmentation preset versus no augmentation. Architecture held fixed at the maxpool baseline. The hypothesis from notebook 6 is that the classification ceiling is data limited, not architecture limited, so augmentation in particular has a real chance of pushing past the 0.625 we have been stuck at.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/AppliedDL/face_age.zip" /content/
!cp -r "/content/drive/MyDrive/Colab Notebooks/AppliedDL/data_splits" /content/
!unzip -q /content/face_age.zip -d /content/

## 2. Imports

In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

cuda
gpu: NVIDIA A100-SXM4-80GB


## 3. Dataset class

Carried from notebook 6, classification only. Added a `channels` argument so the same class works for 1 channel grayscale and 3 channel RGB without duplicating code.

In [4]:
CATEGORIES = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


class FaceAgeDataset(Dataset):
    def __init__(self, csv_path, transform=None, channels=3):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.channels = channels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # convert mode depends on channel count, L for grayscale, RGB for colour
        mode = "L" if self.channels == 1 else "RGB"
        img = Image.open(f"/content/{row['path']}").convert(mode)
        if self.transform:
            img = self.transform(img)
        lbl = torch.tensor(CAT_TO_IDX[row["age_category"]], dtype=torch.long)
        return img, lbl

## 4. Train and eval functions



In [5]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    n_samples = 0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = loss_fn(preds, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)
    return total_loss / n_samples


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    n_samples = 0
    correct = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, lbls)
            total_loss += loss.item() * imgs.size(0)
            n_samples += imgs.size(0)
            correct += (preds.argmax(1) == lbls).sum().item()
    return total_loss / n_samples, correct / n_samples


def run_variant(model, train_loader, val_loader, loss_fn,
                max_epochs=20, patience=5, lr=1e-3, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_acc = -float("inf")
    best_state = None
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    no_improve = 0

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"epoch {epoch+1:2d}  train {train_loss:.4f}  val {val_loss:.4f}  acc {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"early stop at epoch {epoch+1}")
                break

    return {"best_acc": best_acc, "best_state": best_state, "history": history}

## 5. Baseline model

Same maxpool baseline used since notebook 3. Only change, `in_channels` is configurable so the 1 channel variant can use it.

In [6]:
class BaselineCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 25 * 25, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## 6. Transforms

Three transform pipelines, one per variant. ImageNet normalisation for 3 channel, single channel mean and std for 1 channel (computed by averaging the ImageNet stats). Augmentation preset is horizontal flip, small rotation, mild colour jitter, kept moderate since the brief is about whether augmentation helps not which augmentation helps.

In [7]:
# 3 channel, no augmentation, the reference
tf_3ch = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 1 channel, no augmentation
tf_1ch = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.449], std=[0.226]),
])

# 3 channel with moderate augmentation, train only
tf_3ch_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

## 7. Variants

Three variants. `3ch_noaug` is the reference, matches the baseline from earlier notebooks. `1ch_noaug` tests whether colour matters. `3ch_aug` tests whether moderate augmentation pushes past the 0.625 ceiling. Note val loader for the augmentation variant uses the non augmented transform, augmentation is training only.

In [8]:
variants = [
    {"name": "3ch_noaug", "channels": 3, "train_tf": tf_3ch,     "val_tf": tf_3ch},
    {"name": "1ch_noaug", "channels": 1, "train_tf": tf_1ch,     "val_tf": tf_1ch},
    {"name": "3ch_aug",   "channels": 3, "train_tf": tf_3ch_aug, "val_tf": tf_3ch},
]

## 8. Run

In [9]:
results = {}
loss_fn = nn.CrossEntropyLoss()

for v in variants:
    print(f"\n=== {v['name']} ===")
    t0 = time.time()

    train_ds = FaceAgeDataset("/content/data_splits/train.csv",
                              transform=v["train_tf"], channels=v["channels"])
    val_ds = FaceAgeDataset("/content/data_splits/val.csv",
                            transform=v["val_tf"], channels=v["channels"])
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

    model = BaselineCNN(in_channels=v["channels"]).to(device)
    out = run_variant(model, train_loader, val_loader, loss_fn)

    out["time"] = time.time() - t0
    results[v["name"]] = out
    print(f"best acc {out['best_acc']:.4f}  time {out['time']:.1f}s")


=== 3ch_noaug ===
epoch  1  train 1.4802  val 1.1906  acc 0.5150
epoch  2  train 1.0273  val 1.0910  acc 0.5594
epoch  3  train 0.8151  val 0.9883  acc 0.5902
epoch  4  train 0.6663  val 1.0385  acc 0.5970
epoch  5  train 0.5213  val 1.0947  acc 0.6134
epoch  6  train 0.3743  val 1.2551  acc 0.6052
epoch  7  train 0.2556  val 1.6851  acc 0.6038
epoch  8  train 0.1712  val 1.7177  acc 0.6025
epoch  9  train 0.1267  val 2.0034  acc 0.5963
epoch 10  train 0.0950  val 2.3128  acc 0.6059
early stop at epoch 10
best acc 0.6134  time 138.8s

=== 1ch_noaug ===
epoch  1  train 1.4688  val 1.1610  acc 0.5137
epoch  2  train 1.0443  val 1.0030  acc 0.5949
epoch  3  train 0.8765  val 1.0938  acc 0.5697
epoch  4  train 0.7415  val 0.9557  acc 0.5922
epoch  5  train 0.6345  val 0.9675  acc 0.6257
epoch  6  train 0.5154  val 1.0792  acc 0.6011
epoch  7  train 0.3993  val 1.3075  acc 0.5710
epoch  8  train 0.2915  val 1.4156  acc 0.6045
epoch  9  train 0.2260  val 1.6356  acc 0.6004
epoch 10  train 0